In [5]:
import findspark
findspark.init()

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import asc

# spark配置信息
from pyspark import SparkConf

SPARK_APP_NAME = "Action_1"

conf = SparkConf()    # 创建spark config对象
config = (
    ("spark.app.name", SPARK_APP_NAME),    # 设置启动的spark的app名称，没有提供，将随机产生一个名称
    ("spark.executor.memory", "10g"),    # 设置该app启动时占用的内存用量，默认1g
    ("spark.executor.cores", "4"),    # 设置spark executor使用的CPU核心数
)

conf.setAll(config)

# 利用config对象，创建spark session
spark = SparkSession.builder.config(conf=conf).getOrCreate()

# 读取筛选后的数据为DataFrame
action = spark.read.csv("hdfs://localhost:9000//E_commerce_platform/action/filtered_data.csv", header=True, inferSchema=True)
action.show(10)

+-------+--------+------+-------------------+
|item_id| user_id|action|              vtime|
+-------+--------+------+-------------------+
|6289870|u3427707| click|2013-08-30 09:27:15|
|4120848|u3427707| click|2013-05-22 16:54:34|
|4120848|u3427707| click|2013-05-24 12:27:31|
|4690166|u3427707| click|2013-05-22 16:55:10|
|4690166|u3427707| click|2013-05-24 12:27:39|
|4690166|u3427707| click|2013-05-22 16:54:34|
|6301705|u3427707| click|2013-07-29 16:43:02|
|6301705|u3427707| click|2013-07-29 16:54:39|
|6025858|u3427707| click|2013-06-04 12:35:32|
|1528104|u5853382| click|2013-09-16 13:10:44|
+-------+--------+------+-------------------+
only showing top 10 rows



In [15]:
from pyspark.sql.functions import regexp_replace
# 删除rater_uid字段中的'u'
action = action.withColumn('user_id', regexp_replace('user_id', 'u', ''))
# 显示处理后的DataFrame前5行
action.show(5)

+-------+-------+------+-------------------+
|item_id|user_id|action|              vtime|
+-------+-------+------+-------------------+
|6289870|3427707| click|2013-08-30 09:27:15|
|4120848|3427707| click|2013-05-22 16:54:34|
|4120848|3427707| click|2013-05-24 12:27:31|
|4690166|3427707| click|2013-05-22 16:55:10|
|4690166|3427707| click|2013-05-24 12:27:39|
+-------+-------+------+-------------------+
only showing top 5 rows



In [16]:
# 对item_id列进行升序排序
action = action.orderBy(asc('item_id'))
# 显示排序后的结果
action.show(5)

+-------+-------+------+-------------------+
|item_id|user_id|action|              vtime|
+-------+-------+------+-------------------+
|    163|2657764| click|2013-06-30 14:34:30|
|    163|3065348| click|2013-06-29 14:35:26|
|    163|8806288| click|2013-07-04 16:33:15|
|    163|7335561| click|2013-07-03 17:06:08|
|    163|5380909| click|2013-07-11 00:21:49|
+-------+-------+------+-------------------+
only showing top 5 rows



In [22]:
from pyspark.sql.functions import col, expr, from_unixtime
from datetime import datetime


# 将时间字符串转换为时间戳
action = action.withColumn("vtime", from_unixtime(col("vtime"), 'yyyy-MM-dd HH:mm:ss'))
action = action.withColumn("timestamp", expr("UNIX_TIMESTAMP(vtime, 'yyyy-MM-dd HH:mm:ss')"))

# 计算当前时间
current_time = int(datetime.now().timestamp())

# 时间序列打分方法
decay_rate = 0.0001

action = action.withColumn("time_diff", current_time - col("timestamp"))
action = action.withColumn("decay_factor", expr(f"EXP(-{decay_rate} * time_diff)"))
action = action.withColumn("time_series_score", col("decay_factor"))
action.show(5)

+-------+--------+------+-------------------+----------+---------+------------+-----------------+
|item_id| user_id|action|              vtime| timestamp|time_diff|decay_factor|time_series_score|
+-------+--------+------+-------------------+----------+---------+------------+-----------------+
|    163|12394550| click|2013-07-08 13:55:05|1373262905|340418421|         0.0|              0.0|
|    163|13811342| click|2013-07-08 13:35:35|1373261735|340419591|         0.0|              0.0|
|    163| 3228928| click|2013-07-11 17:46:32|1373535992|340145334|         0.0|              0.0|
|    163|11904148| click|2013-06-29 16:51:38|1372495898|341185428|         0.0|              0.0|
|    163| 2868413| click|2013-06-23 10:22:14|1371954134|341727192|         0.0|              0.0|
+-------+--------+------+-------------------+----------+---------+------------+-----------------+
only showing top 5 rows



In [24]:
# 偏好加权打分方法
preference_weights = {
    "click": 0.2,
    "collect": 0.4,
    "cart": 0.6,
    "alipay": 1.0
}

action = action.withColumn("preference_weighted_score", expr(
    "CASE WHEN action IN ('alipay', 'cart', 'collect', 'click') " +
    f"THEN CASE action " +
    f"WHEN 'alipay' THEN {preference_weights['alipay']} " +
    f"WHEN 'cart' THEN {preference_weights['cart']} " +
    f"WHEN 'collect' THEN {preference_weights['collect']} " +
    f"WHEN 'click' THEN {preference_weights['click']} " +
    f"END " +
    "ELSE 0.0 END"
))

action.show(5)

+-------+--------+------+-------------------+----------+---------+------------+-----------------+-------------------------+
|item_id| user_id|action|              vtime| timestamp|time_diff|decay_factor|time_series_score|preference_weighted_score|
+-------+--------+------+-------------------+----------+---------+------------+-----------------+-------------------------+
|    163| 2657764| click|2013-06-30 14:34:30|1372574070|341107256|         0.0|              0.0|                      0.2|
|    163|13811342| click|2013-07-08 13:35:35|1373261735|340419591|         0.0|              0.0|                      0.2|
|    163| 3228928| click|2013-07-11 17:46:32|1373535992|340145334|         0.0|              0.0|                      0.2|
|    163| 3724649| click|2013-06-28 09:10:14|1372381814|341299512|         0.0|              0.0|                      0.2|
|    163| 2620101| click|2013-06-26 16:30:17|1372235417|341445909|         0.0|              0.0|                      0.2|
+-------

In [26]:
# 加权得分

action = action.withColumn("weighted_score", expr(
    f"IF(time_series_score != 0.0 AND preference_weighted_score != 0.0, " +
    "time_series_score * preference_weighted_score, preference_weighted_score)"
))
action.show(5)

+-------+--------+------+-------------------+----------+---------+------------+-----------------+-------------------------+--------------+
|item_id| user_id|action|              vtime| timestamp|time_diff|decay_factor|time_series_score|preference_weighted_score|weighted_score|
+-------+--------+------+-------------------+----------+---------+------------+-----------------+-------------------------+--------------+
|    163| 8806288| click|2013-07-04 16:33:15|1372926795|340754531|         0.0|              0.0|                      0.2|           0.2|
|    163| 2657764| click|2013-06-30 14:34:30|1372574070|341107256|         0.0|              0.0|                      0.2|           0.2|
|    163| 3065348| click|2013-06-29 14:35:26|1372487726|341193600|         0.0|              0.0|                      0.2|           0.2|
|    163|13811342| click|2013-07-08 13:35:35|1373261735|340419591|         0.0|              0.0|                      0.2|           0.2|
|    163| 3228928| click|20

In [27]:
# 删除列
action = action.drop('action','vtime','timestamp','time_diff','decay_factor','time_series_score','preference_weighted_score')

# 显示删除列后的DataFrame
action.show(10)

+-------+--------+--------------+
|item_id| user_id|weighted_score|
+-------+--------+--------------+
|    163|10296125|           0.2|
|    163| 8316348|           0.2|
|    163| 6734859|           0.2|
|    163| 5094293|           0.2|
|    163| 4889126|           0.2|
|    163| 9767215|           0.2|
|    163| 7615221|           0.2|
|    163|11019638|           0.2|
|    163|11019638|           0.2|
|    163| 7382230|           0.2|
+-------+--------+--------------+
only showing top 10 rows



In [29]:
from pyspark.ml.recommendation import ALS

# 将列的数据类型转换为数字类型
action = action.withColumn("weighted_score", action["weighted_score"].cast("double"))
action = action.withColumn("item_id", action["item_id"].cast("int"))
action = action.withColumn("user_id", action["user_id"].cast("int"))

# 初始化ALS模型
als = ALS(maxIter=5, regParam=0.01, userCol="user_id", itemCol="item_id", ratingCol="weighted_score",
          coldStartStrategy="drop")
model = als.fit(action)

Py4JJavaError: An error occurred while calling o391.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 6 in stage 28.0 failed 1 times, most recent failure: Lost task 6.0 in stage 28.0 (TID 842, LAPTOP-L36Q0L8U, executor driver): java.lang.OutOfMemoryError: Java heap space
	at java.lang.reflect.Array.newInstance(Array.java:75)
	at java.io.ObjectInputStream.readArray(ObjectInputStream.java:2124)
	at java.io.ObjectInputStream.readObject0(ObjectInputStream.java:1698)
	at java.io.ObjectInputStream.defaultReadFields(ObjectInputStream.java:2472)
	at java.io.ObjectInputStream.readSerialData(ObjectInputStream.java:2396)
	at java.io.ObjectInputStream.readOrdinaryObject(ObjectInputStream.java:2254)
	at java.io.ObjectInputStream.readObject0(ObjectInputStream.java:1710)
	at java.io.ObjectInputStream.readObject(ObjectInputStream.java:508)
	at java.io.ObjectInputStream.readObject(ObjectInputStream.java:466)
	at org.apache.spark.serializer.JavaDeserializationStrea

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 2075)
Traceback (most recent call last):
  File "F:\Anaconda\procedure\lib\socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "F:\Anaconda\procedure\lib\socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "F:\Anaconda\procedure\lib\socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "F:\Anaconda\procedure\lib\socketserver.py", line 747, in __init__
    self.handle()
  File "D:\home\spark\spark-3.0.0-bin-hadoop2.7\spark-3.0.0-bin-hadoop2.7\python\pyspark\accumulators.py", line 268, in handle
    poll(accum_updates)
  File "D:\home\spark\spark-3.0.0-bin-hadoop2.7\spark-3.0.0-bin-hadoop2.7\python\pyspark\accumulators.py", line 241, in poll
    if func():
  File "D:\home\spark\spark-3.0.0-bin-hadoop2.7\spark-3.0.0

In [ ]:
userRecs = model.recommendForAllUsers(5)  # 推荐每个用户前5个物品
userRecs.show(10)

## 评估

In [ ]:
# 在测试集上进行预测
predictions = model.transform(action)
predictions.show()

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

# 定义评估器（使用均方根误差和平均绝对误差）
evaluator_rmse = RegressionEvaluator(metricName="rmse", labelCol="weighted_score", predictionCol="prediction")
evaluator_mae = RegressionEvaluator(metricName="mae", labelCol="weighted_score", predictionCol="prediction")

# 计算均方根误差
rmse = evaluator_rmse.evaluate(predictions)
print("均方根误差 (RMSE) = {:.4f}".format(rmse))

# 计算平均绝对误差
mae = evaluator_mae.evaluate(predictions)
print("平均绝对误差 (MAE) = {:.4f}".format(mae))

## 保存

In [ ]:
# 将数组中的struct字段展开为列
expanded_df = userRecs.select('user_id', 'recommendations.item_id', 'recommendations.rating')

# 将DataFrame转换为Pandas DataFrame
pandas_df = expanded_df.toPandas()
# 保存为 CSV 文件
pandas_df.to_csv('D:\\Datas\\datas\\User_recommendations.csv', index=False)